# Tutorial: Build Impact Figure

- Rebuild the manuscript's citation-impact figure from the local derived parquet file.

Prerequisites:
- Run `1.0.0-prepare_avg_citations.ipynb` first, or otherwise make sure `workspace/data/derived/avg_citations_result.parquet` exists.
- Install the Python dependencies from `../config/requirements.txt`.


## Outline

1. Resolve the companion config and the figure input/output paths.
2. Load the derived parquet file and isolate the highlighted Bertone & Hooper paper.
3. Build the scatter plot step by step.
4. Export the finished figure to the manuscript output directory.


In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path

def _find_workspace_root() -> Path:
    env_root = os.environ.get("WORKSPACE_ROOT", "").strip()
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    for start in [Path.cwd(), *Path.cwd().parents]:
        if (start / "config" / "workspace.json").exists():
            return start
    raise FileNotFoundError("Could not locate workspace root from WORKSPACE_ROOT or the current working directory.")

WORKSPACE_DIR = _find_workspace_root()
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
PAPER_DIR = paths["paper_dir"]
PAPER_OUTPUTS_DIR = paths["paper_outputs"]
OVERLEAF_DIR = paths["overleaf"]
SHARED_ASSETS_DIR = paths["shared_assets"]

print("workspace_root:", WORKSPACE_DIR)
print("shared_assets_root:", SHARED_ASSETS_DIR)

MANUSCRIPT_FIGURES_DIR = PAPER_DIR / "assets" / "figures"
OVERLEAF_FIGURES_DIR = OVERLEAF_DIR / "figures"
for _path in [MANUSCRIPT_FIGURES_DIR, OVERLEAF_FIGURES_DIR]:
    _path.mkdir(parents=True, exist_ok=True)


def sync_manuscript_figure(source: Path, target_name: str | None = None) -> None:
    source = Path(source)
    name = target_name or source.name
    paper_target = MANUSCRIPT_FIGURES_DIR / name
    overleaf_target = OVERLEAF_FIGURES_DIR / name
    shutil.copy2(source, paper_target)
    shutil.copy2(source, overleaf_target)


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import yaml

CONFIG_PATH = CONFIG_DIR / "paths.yml"

def load_parquet_frame(path: Path) -> pd.DataFrame:
    """Read project parquet files through pyarrow to avoid pandas/pyarrow extension hotfix issues."""
    return pq.read_table(path).to_pandas()

CONFIG_PATH

## Step 1 - Resolve the configured paths

This mirrors the build config so the notebook and the automated workflow stay aligned.


In [ ]:
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

def resolve_path(specification):
    path = Path(str(specification)).expanduser()
    if path.is_absolute():
        return path
    return (WORKSPACE_DIR / path).resolve()

input_parquet_path = resolve_path(config["outputs"]["avg_citations_parquet"])
output_pdf_path = resolve_path(config["outputs"]["impact_figure_pdf"])
font_path = resolve_path(config["inputs"]["libertine_font"])

pd.Series(
    {
        "input_parquet": str(input_parquet_path),
        "output_pdf": str(output_pdf_path),
        "font_path": str(font_path),
    }
)


## Step 2 - Load the derived ranking data

Because the canonical local sources do not carry title and author metadata, the figure annotation focuses on the bibcode and citation-rank statistics.


In [ ]:
result = load_parquet_frame(input_parquet_path)
filtered_result = result[result["avg_citations_per_year"] > 0].copy()

target_bibcode = "2018RvMP...90d5002B"
target_paper = filtered_result[filtered_result["bibcode"] == target_bibcode].iloc[0]
overall_mean = filtered_result["avg_citations_per_year"].mean()
overall_median = filtered_result["avg_citations_per_year"].median()

target_paper


## Step 3 - Build the figure

This cell configures the font if the project-local Libertine file is available, adds reproducible jitter, and assembles the plot styling in one place.


In [ ]:
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.size"] = 14
if font_path.exists():
    fm.fontManager.addfont(str(font_path))
    mpl.rcParams["font.family"] = fm.FontProperties(fname=str(font_path)).get_name()
else:
    mpl.rcParams["font.family"] = "DejaVu Serif"

rng = np.random.default_rng(42)
y_jitter = rng.uniform(-1.5, 1.5, len(filtered_result))
x_jitter = rng.uniform(-1.5, 1.5, len(filtered_result))

fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(
    filtered_result["avg_citations_per_year"] + x_jitter,
    filtered_result["percentile"] + y_jitter,
    alpha=0.15,
    color="#FFA239",
    marker="2",
    linewidths=0.2,
    edgecolors="black",
    s=10,
)
ax.scatter(
    target_paper["avg_citations_per_year"],
    target_paper["percentile"],
    color="#E37434",
    s=50,
    marker="o",
    edgecolors="black",
    linewidths=0.5,
    label=target_bibcode,
    zorder=5,
)

ax.axvline(overall_mean, color="#2cb869", linestyle="--", linewidth=1.5, label="Overall mean")
ax.axvline(overall_median, color="#333333", linestyle="--", linewidth=1.5, label="Overall median")
ax.annotate(
    (
        f"Bibcode: {target_paper['bibcode']}\n"
        f"Avg cit/year: {target_paper['avg_citations_per_year']:.2f}\n"
        f"Percentile: {target_paper['percentile']:.1f}"
    ),
    (target_paper["avg_citations_per_year"], target_paper["percentile"]),
    xytext=(-113, -80),
    textcoords="offset points",
    fontsize=16,
    color="#333333",
    bbox=dict(boxstyle="round", fc="white", ec="#333333"),
    arrowprops=dict(color="#333333", arrowstyle="->", lw=1.0),
)

ax.set_xscale("symlog")
ax.set_xlim(1, 550)
ax.xaxis.set_major_formatter(
    ticker.FuncFormatter(
        lambda value, pos: f"{int(value):,}" if value >= 1 else (f"{value:.1f}" if value > 0 else "")
    )
)
ax.set_xticks([1, 2, 5, 10, 25, 50, 100, 200, 500])
ax.tick_params(axis="x", which="major", labelsize=14)
ax.tick_params(axis="y", which="major", labelsize=14)
ax.grid(True, linestyle="--", alpha=0.6)
ax.set_title("Percentile vs. average citations per year", fontsize=20)
ax.set_xlabel("Average citations per year (symlog)", fontsize=18)
ax.set_ylabel("Percentile", fontsize=18)
ax.legend(fontsize=14)
fig.tight_layout()



## Step 4 - Export the manuscript figure

The notebook writes `fig-impact.pdf` under `workspace/outputs/manuscript/figures/` and immediately syncs the fresh file into `../paper/assets/figures/` and `../overleaf/figures/`.


In [ ]:
output_pdf_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_pdf_path, bbox_inches="tight")
sync_manuscript_figure(output_pdf_path)
output_pdf_path
